# Python for Finance: Historical Volatility & Risk-Return Measures

<b> YouTube Tutorial </b> (Published: May 11, 2021): https://youtu.be/j4c2XqiJzRU

In this tutorial we compute and track historical volatility over time.

In [ ]:
## This is required for pandas_datareader on google colab - then you need to restart runtime
!pip install --upgrade pandas_datareader

In [1]:
import datetime as dt
import pandas as pd
import numpy as np

#from pandas_datareader import data as pdr
import yfinance as pdr
import plotly.offline as pyo
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pyo.init_notebook_mode(connected=True)
pd.options.plotting.backend = 'plotly'

### Get stock data with pandas_datareader

In [2]:
end = dt.datetime.now()
start = dt.datetime(2015,1,1)

df = pdr.download(['^AXJO', 'CBA.AX','NAB.AX','STO.AX'], start, end)
Close = df.Close
Close.head()

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  4 of 4 completed


Ticker,CBA.AX,NAB.AX,STO.AX,^AXJO
Date,,,,
2015-01-02,52.783749,17.686415,5.267106,5435.899902
2015-01-05,52.913033,17.712769,5.312125,5450.299805
2015-01-06,52.512867,17.570475,4.855514,5364.799805
2015-01-07,52.395916,17.612635,4.791203,5353.600098
2015-01-08,52.568295,17.770737,4.694735,5381.500000


### Compute log returns

In [3]:
log_returns = np.log(df.Close/df.Close.shift(1)).dropna()
log_returns

Ticker,CBA.AX,NAB.AX,STO.AX,^AXJO
Date,,,,
2015-01-05,0.002446,0.001489,0.008511,0.002646
2015-01-06,-0.007591,-0.008066,-0.089877,-0.015812
2015-01-07,-0.002230,0.002397,-0.013334,-0.002090
2015-01-08,0.003285,0.008937,-0.020340,0.005198
2015-01-09,0.007583,0.015011,0.052047,0.015507
...,...,...,...,...
2025-07-21,-0.025478,-0.024278,0.002561,-0.010215
2025-07-22,-0.031120,-0.027297,0.006373,0.001038
2025-07-23,0.005091,-0.000538,-0.007653,0.006891


### Calculate daily standard deviation of returns

In [4]:
daily_std = log_returns.std()
daily_std

Ticker
CBA.AX    0.013557
NAB.AX    0.014102
STO.AX    0.025483
^AXJO     0.009646
dtype: float64

In [5]:
annualized_std = daily_std * np.sqrt(252)
annualized_std

Ticker
CBA.AX    0.215209
NAB.AX    0.223863
STO.AX    0.404537
^AXJO     0.153125
dtype: float64

### Plot histogram of log returns with annualized volatility

In [6]:
fig = make_subplots(rows=2, cols=2)

trace0 = go.Histogram(x=log_returns['CBA.AX'], name='CBA')
trace1 = go.Histogram(x=log_returns['NAB.AX'], name='NAB')
trace2 = go.Histogram(x=log_returns['STO.AX'], name='STO')
#trace3 = go.Histogram(x=log_returns['WPL.AX'], name='WPL')

fig.append_trace(trace0, 1, 1)
fig.append_trace(trace1, 1, 2)
fig.append_trace(trace2, 2, 1)
#fig.append_trace(trace3, 2, 2)

fig.update_layout(autosize = False, width=700, height=600, title='Frequency of log returns',
                  xaxis=dict(title='CBA Annualized Volatility: ' + str(np.round(annualized_std['CBA.AX']*100, 1))),
                  xaxis2=dict(title='NAB Annualized Volatility: ' + str(np.round(annualized_std['NAB.AX']*100, 1))),
                  xaxis3=dict(title='STO Annualized Volatility: ' + str(np.round(annualized_std['STO.AX']*100, 1))),
                  #xaxis4=dict(title='WPL Annualized Volatility: ' + str(np.round(annualized_std['WPL.AX']*100, 1)))
                  )

fig.show(renderer="colab")

In [7]:
TRADING_DAYS = 60
volatility = log_returns.rolling(window=TRADING_DAYS).std()*np.sqrt(TRADING_DAYS)
volatility.tail()

Ticker,CBA.AX,NAB.AX,STO.AX,^AXJO
Date,,,,
2025-07-21,0.083713,0.083289,0.141502,0.038621
2025-07-22,0.088909,0.086846,0.140441,0.038556
2025-07-23,0.089005,0.086653,0.140261,0.038156
2025-07-24,0.086420,0.087303,0.139679,0.037995
2025-07-25,0.086454,0.087325,0.138430,0.038450


In [8]:
volatility.plot().update_layout(autosize = False, width=600, height=300).show(renderer="colab")

### Sharpe ratio
The Sharpe ratio which was introduced in 1966 by Nobel laureate William F. Sharpe is a measure for calculating risk-adjusted return. The Sharpe ratio is the average return earned in excess of the risk-free rate per unit of volatility.

In [9]:
Rf = 0.01/255
sharpe_ratio = (log_returns.rolling(window=TRADING_DAYS).mean() - Rf)*TRADING_DAYS / volatility

In [10]:
sharpe_ratio.plot().update_layout(autosize = False, width=600, height=300).show(renderer="colab")

#### Sortino Ratio
The Sortino ratio is very similar to the Sharpe ratio, the only difference being that where the Sharpe ratio uses all the observations for calculating the standard deviation the Sortino ratio only considers the harmful variance.

In [11]:
sortino_vol = log_returns[log_returns<0].rolling(window=TRADING_DAYS, center=True, min_periods=10).std()*np.sqrt(TRADING_DAYS)
sortino_ratio = (log_returns.rolling(window=TRADING_DAYS).mean() - Rf)*TRADING_DAYS / sortino_vol

In [12]:
sortino_vol.plot().update_layout(autosize = False, width=600, height=300).show(renderer="colab")

In [13]:
sortino_ratio.plot().update_layout(autosize = False, width=600, height=300).show(renderer="colab")

### Modigliani ratio (M2 ratio)

The Modigliani ratio measures the returns of the portfolio, adjusted for the risk of the portfolio relative to that of some benchmark.

In [14]:
m2_ratio = pd.DataFrame()

benchmark_vol = volatility['^AXJO']
for c in log_returns.columns:
    if c != '^AXJO':
        m2_ratio[c] = (sharpe_ratio[c]*benchmark_vol/TRADING_DAYS + Rf)*TRADING_DAYS

In [15]:
m2_ratio.plot().update_layout(autosize = False, width=600, height=300).show(renderer="colab")

### Max Drawdown

Max drawdown quantifies the steepest decline from peak to trough observed for an investment. This is useful for a number of reasons, mainly the fact that it doesn't rely on the underlying returns being normally distributed.

In [16]:
def max_drawdown(returns):
    cumulative_returns = (returns+1).cumprod()
    peak = cumulative_returns.expanding(min_periods=1).max()
    drawdown = (cumulative_returns/peak)-1
    return drawdown.min()


returns = df.Close.pct_change()
max_drawdowns = returns.apply(max_drawdown, axis=0)
max_drawdowns*100

C:\Users\grant\AppData\Local\Temp\ipykernel_18304\722799521.py:8: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



Ticker
CBA.AX   -39.015783
NAB.AX   -52.232507
STO.AX   -69.139509
^AXJO    -36.530541
dtype: float64

### Calmar Ratio

Calmar ratio uses max drawdown in the denominator as opposed to standard deviation.

In [17]:
calmars = np.exp(log_returns.mean()*255)/abs(max_drawdowns)
calmars.plot.bar().update_layout(autosize = False, width=600, height=300).show(renderer="colab")